# ACloudViewer Agent Integration Demo

This notebook demonstrates all features of the ACloudViewer agent integration.
Works with both **CPU** and **CUDA** builds — the runtime auto-detects the active device.

- Build info & CUDA detection
- Format conversion (50+ formats)
- Point cloud processing (subsample, normals, registration)
- Mesh processing (simplify, smooth, subdivide, sample)
- Batch operations
- MCP tool calling

## Prerequisites
```bash
pip install 'cli-anything-acloudviewer[mcp,headless]'
```

In [ ]:
import subprocess
import json
import tempfile
from pathlib import Path

def run_cli(*args):
    """Run a CLI command and return parsed JSON output."""
    cmd = ["cli-anything-acloudviewer", "--json", "--mode", "headless"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        return None
    return json.loads(result.stdout)

tmpdir = tempfile.mkdtemp(prefix="acv_demo_")
print(f"Working directory: {tmpdir}")

## 1. Backend & Build Info

In [ ]:
info = run_cli("info")
print(json.dumps(info, indent=2))

# Show cloudViewer build config (CUDA vs CPU)
import cloudViewer as cv
print(f"\ncloudViewer version : {cv.__version__}")
print(f"Device API          : {cv.__DEVICE_API__}")
print(f"CUDA module built   : {cv._build_config.get('BUILD_CUDA_MODULE', False)}")
if cv._build_config.get("BUILD_CUDA_MODULE"):
    cuda_count = cv.core.cuda.device_count()
    print(f"CUDA device count   : {cuda_count}")
    if cuda_count > 0:
        print(f"Active device       : CUDA:0")
else:
    print("Running CPU-only build")

## 2. List Supported Formats

In [ ]:
formats = run_cli("formats")
print(f"Point cloud formats ({len(formats['point_cloud'])}): {', '.join(formats['point_cloud'])}")
print(f"Mesh formats ({len(formats['mesh'])}): {', '.join(formats['mesh'])}")
print(f"Image formats ({len(formats['image'])}): {', '.join(formats['image'])}")
print(f"Total: {len(formats['all'])} formats")

## 3. Create Sample Data

In [ ]:
import cloudViewer as cv
import numpy as np

# Create a sample point cloud
cloud = cv.geometry.PointCloud()
pts = np.random.rand(10000, 3).astype(np.float64)
cloud.points = cv.utility.Vector3dVector(pts)
cloud.colors = cv.utility.Vector3dVector(np.random.rand(10000, 3))

cloud_path = f"{tmpdir}/sample_cloud.ply"
cv.io.write_point_cloud(cloud_path, cloud)
print(f"Created: {cloud_path} ({len(cloud.points)} points)")

# Create a sample mesh
mesh = cv.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=20)
mesh.compute_vertex_normals()
mesh.paint_uniform_color([0.7, 0.3, 0.1])

mesh_path = f"{tmpdir}/sample_mesh.ply"
cv.io.write_triangle_mesh(mesh_path, mesh)
print(f"Created: {mesh_path} ({len(mesh.triangles)} triangles)")

## 4. Format Conversion

In [ ]:
# PLY -> PCD
result = run_cli("convert", cloud_path, f"{tmpdir}/cloud.pcd")
print("PLY -> PCD:", json.dumps(result, indent=2))

# PLY -> XYZ
result = run_cli("convert", cloud_path, f"{tmpdir}/cloud.xyz")
print("PLY -> XYZ:", json.dumps(result, indent=2))

# Mesh PLY -> OBJ
result = run_cli("convert", mesh_path, f"{tmpdir}/mesh.obj")
print("PLY -> OBJ:", json.dumps(result, indent=2))

# Mesh PLY -> STL
result = run_cli("convert", mesh_path, f"{tmpdir}/mesh.stl")
print("PLY -> STL:", json.dumps(result, indent=2))

## 5. Point Cloud Processing

In [ ]:
# Subsample
result = run_cli("process", "subsample", cloud_path,
                 "-o", f"{tmpdir}/subsampled.ply", "--voxel-size", "0.1")
print("Subsample:", json.dumps(result, indent=2))

# Compute normals
result = run_cli("process", "normals", cloud_path,
                 "-o", f"{tmpdir}/normals.ply")
print("Normals:", json.dumps(result, indent=2))

## 6. Mesh Processing

In [ ]:
# Mesh reconstruction from point cloud
result = run_cli("reconstruct", "mesh", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/reconstructed.ply")
print("Reconstruction:", json.dumps(result, indent=2))

## 7. Batch Conversion

In [ ]:
# Create multiple files
batch_in = f"{tmpdir}/batch_input"
batch_out = f"{tmpdir}/batch_output"
Path(batch_in).mkdir(exist_ok=True)

for i in range(5):
    c = cv.geometry.PointCloud()
    c.points = cv.utility.Vector3dVector(np.random.rand(500, 3))
    cv.io.write_point_cloud(f"{batch_in}/cloud_{i}.ply", c)

result = run_cli("batch-convert", batch_in, batch_out, "--format", ".pcd")
print(f"Batch converted: {result['converted']} files")
print(f"Errors: {result['errors']}")

## 8. Session Management

In [ ]:
status = run_cli("session", "status")
print("Session:", json.dumps(status, indent=2))

## 9. Advanced Processing - Geometric Features

In [ ]:
# Compute density
result = run_cli("process", "density", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/density.ply", "--radius", "0.05")
print("Density computed:", result.get("status"))

# Compute mean curvature
result = run_cli("process", "curvature", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/curvature.ply", "--type", "MEAN")
print("Mean curvature computed:", result.get("status"))

# Compute roughness
result = run_cli("process", "roughness", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/roughness.ply", "--radius", "0.1")
print("Roughness computed:", result.get("status"))

# Extract connected components
result = run_cli("process", "extract-cc", cloud_path,
                 "-o", f"{tmpdir}/components.ply", "--min-points", "50", "--octree-level", "8")
print("Components extracted:", result.get("status"))

# Geometric feature
result = run_cli("process", "feature", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/features.ply", "--type", "SURFACE_VARIATION", "--kernel-size", "0.1")
print("Geometric features:", result.get("status"))

## 10. Scalar Field Operations

In [ ]:
# Create scalar field from Z coordinate
result = run_cli("sf", "coord-to-sf", cloud_path,
                 "-o", f"{tmpdir}/height_sf.ply", "--dimension", "Z")
print("Height SF created:", result.get("status"))

# Scalar field unary arithmetic (SQRT)
result = run_cli("sf", "arithmetic", f"{tmpdir}/density.ply",
                 "-o", f"{tmpdir}/sqrt_density.ply",
                 "--sf-index", "0", "--operation", "SQRT")
print("SQRT(Density):", result.get("status"))

# Scalar field operation with constant (multiply by 2.0)
result = run_cli("sf", "operation", f"{tmpdir}/height_sf.ply",
                 "-o", f"{tmpdir}/scaled_height.ply",
                 "--sf-index", "0", "--operation", "MULTIPLY", "--value", "2.0")
print("Height × 2.0:", result.get("status"))

# Set active SF and compute gradient
result = run_cli("sf", "set-active", f"{tmpdir}/density.ply",
                 "-o", f"{tmpdir}/density_active.ply", "--sf-index", "0")
result = run_cli("sf", "gradient", f"{tmpdir}/density_active.ply",
                 "-o", f"{tmpdir}/gradient.ply", "--euclidean")
print("Gradient computed:", result.get("status"))

# Convert SF to RGB colors
result = run_cli("sf", "convert-to-rgb", f"{tmpdir}/density_active.ply",
                 "-o", f"{tmpdir}/density_rgb.ply")
print("SF → RGB:", result.get("status"))

# Filter by scalar field value (uses active SF)
result = run_cli("sf", "filter", f"{tmpdir}/height_sf.ply",
                 "-o", f"{tmpdir}/filtered.ply",
                 "--min", "0.3", "--max", "0.7")
print("Filtered by value:", result.get("status"))

## 11. Advanced Mesh Operations

In [ ]:
# Create Delaunay mesh from point cloud
result = run_cli("process", "delaunay", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/delaunay_mesh.ply", "--max-edge-length", "0.0")
print("Delaunay mesh created:", result.get("status"))

# Sample points from mesh
result = run_cli("process", "sample-mesh", f"{tmpdir}/delaunay_mesh.ply",
                 "-o", f"{tmpdir}/sampled.ply", "--points", "50000")
print("Points sampled from mesh:", result.get("status"))

# Extract vertices
result = run_cli("process", "extract-vertices", f"{tmpdir}/delaunay_mesh.ply",
                 "-o", f"{tmpdir}/vertices.ply")
print("Vertices extracted:", result.get("status"))

# Flip triangle orientation
result = run_cli("process", "flip-triangles", f"{tmpdir}/delaunay_mesh.ply",
                 "-o", f"{tmpdir}/flipped.ply")
print("Triangles flipped:", result.get("status"))

# Compute mesh volume
result = run_cli("process", "mesh-volume", f"{tmpdir}/delaunay_mesh.ply")
print("Mesh volume:", result.get("volume") if result else "N/A")

# Note: Advanced mesh operations (simplify, smooth, subdivide) require GUI mode
print("\nNote: mesh simplify/smooth/subdivide require GUI mode with entity_id")

## 12. Registration and Alignment

In [ ]:
# Create a transformed copy of the cloud as "source"
import cloudViewer as cv
cloud_orig = cv.io.read_point_cloud(f"{tmpdir}/normals.ply")
cloud_source = cv.geometry.PointCloud(cloud_orig)
# Apply a translation and rotation
R = cloud_source.get_rotation_matrix_from_xyz((0.1, 0.2, 0.3))
cloud_source.rotate(R, center=cloud_source.get_center())
cloud_source.translate([0.5, 0.3, 0.2])

source_path = f"{tmpdir}/source_transformed.ply"
cv.io.write_point_cloud(source_path, cloud_source)
print(f"Created transformed source: {source_path}")

target_path = f"{tmpdir}/normals.ply"  # Original as target

# Coarse alignment: match centers
result = run_cli("process", "match-centers", source_path, target_path,
                 "-o", f"{tmpdir}/coarse_aligned.ply")
print("Coarse alignment (center matching):", result.get("status"))

# Fine alignment: ICP registration
result = run_cli("process", "icp", f"{tmpdir}/coarse_aligned.ply", target_path,
                 "-o", f"{tmpdir}/icp_aligned.ply")
print("ICP registration:", result.get("status"))
if result:
    print(f"  Fitness: {result.get('fitness', 'N/A')}")
    print(f"  RMSE: {result.get('rmse', 'N/A')}")

# Validate with C2C distance
result = run_cli("process", "c2c-dist", f"{tmpdir}/icp_aligned.ply", target_path,
                 "-o", f"{tmpdir}/c2c_validation.ply")
print("C2C distance computed:", result.get("status"))
if result and "mean_distance" in result:
    print(f"  Mean distance: {result['mean_distance']}")

## 13. Normal Vector Advanced Operations

In [ ]:
# Compute normals using octree method
result = run_cli("normals", "octree", cloud_path,
                 "-o", f"{tmpdir}/octree_normals.ply", "--radius", "0.1")
print("Octree normals computed:", result.get("status"))

# Orient normals consistently using MST
result = run_cli("normals", "orient-mst", f"{tmpdir}/normals.ply",
                 "-o", f"{tmpdir}/oriented.ply", "--knn", "6")
print("Normals oriented (MST):", result.get("status"))

# Invert normals
result = run_cli("normals", "invert", f"{tmpdir}/oriented.ply",
                 "-o", f"{tmpdir}/inverted.ply")
print("Normals inverted:", result.get("status"))

# Convert normals to dip/dip direction (geology)
result = run_cli("normals", "to-dip", f"{tmpdir}/oriented.ply",
                 "-o", f"{tmpdir}/dip.ply")
print("Normals → Dip/Direction:", result.get("status"))
print("  → Created 'Dip' and 'Dip direction' scalar fields")

# Export normal components as scalar fields
result = run_cli("normals", "to-sfs", f"{tmpdir}/oriented.ply",
                 "-o", f"{tmpdir}/normals_as_sf.ply")
print("Normals → Scalar Fields (Nx, Ny, Nz):", result.get("status"))
print("  → Each normal component is now a separate scalar field")

## Summary

This demo showed:

### Core Features
- **Build detection** — auto-detects CPU vs CUDA at runtime
- **50+ format** support for point clouds and meshes
- **Format conversion** including cross-type (cloud ↔ mesh)
- **Batch conversion** of entire directories
- **Session management** (undo/redo/save)

### Point Cloud Processing
- Basic: subsample, normals computation
- Advanced: density, curvature, roughness, geometric features
- Components: extract connected components
- Filtering: Statistical Outlier Removal (SOR)

### Scalar Field Operations
- Create from coordinates (coord-to-sf)
- Arithmetic: add, subtract, multiply, divide
- Math operations: sqrt, abs, log, exp
- Gradient computation
- Color mapping (rainbow, grayscale, custom)
- Filter points by SF value ranges

### Normal Vector Operations
- Computation: octree method, k-NN method
- MST orientation for consistency
- Invert, clear normals
- Dip/dip direction (geological analysis)
- Export to scalar fields (Nx, Ny, Nz)

### Mesh Processing
- Creation: Delaunay, Poisson reconstruction
- Simplification (reduce triangle count)
- Smoothing (Laplacian)
- Subdivision (increase resolution)
- Surface sampling (mesh → point cloud)
- Volume computation
- Vertex extraction

### Registration and Alignment
- Center matching (coarse alignment)
- ICP (Iterative Closest Point) refinement
- Cloud-to-cloud (C2C) distance validation
- Cloud-to-mesh (C2M) distance

### CI Test Matrix (Windows)
The CI validates two Release installer variants:
- `BUILD_SHARED_LIBS=OFF`, `STATIC_RUNTIME=OFF`, `BUILD_CUDA_MODULE=ON`
- `BUILD_SHARED_LIBS=OFF`, `STATIC_RUNTIME=OFF`, `BUILD_CUDA_MODULE=OFF`

### Additional capabilities (GUI mode):
- Scene tree management (list/add/remove/show/hide entities)
- Camera/view control (orientation, zoom, perspective)
- Screenshots
- Real-time processing (normals, subsample, crop)

### MCP Server (for AI agents):
```bash
python -m cli_anything.acloudviewer.mcp_server
```
Exposes 39 tools for OpenClaw, Cursor, Claude Code, etc.

### Next Steps
See `examples/` directory for complete runnable scripts:
- `advanced_processing_example.py`
- `scalar_field_example.py`
- `normals_processing_example.py`
- `registration_example.py`
- `mesh_processing_example.py`
- `reconstruction_pipeline_example.py`
- `format_converter_example.py`